# 08.01 · RAG from scratch (no framework)

**Chapter:** Retrieval-Augmented Generation · **Milestone:** M3

## Goals
- [ ] Build the four steps yourself: index → retrieve → augment (prompt) → generate
- [ ] Make answers cite their sources, and say "I don't know" when the sources don't contain the answer
- [ ] Measure it: answer accuracy, citation validity, abstention on unanswerable questions

## Resources
- [DataTalksClub LLM Zoomcamp (free course, RAG modules)](https://github.com/DataTalksClub/llm-zoomcamp) — free
- [Lewis et al. – the original RAG paper](https://arxiv.org/abs/2005.11401) — free
- [Anthropic – Reduce hallucinations (quotes & citations)](https://docs.anthropic.com/en/docs/test-and-evaluate/strengthen-guardrails/reduce-hallucinations) — free

**Why RAG**: models don't know your private/recent data, and they hallucinate when asked anyway. RAG puts the relevant facts *in the
prompt*, so the model reads instead of remembers. It's the most common LLM application pattern.

In [ ]:
import inspect
import re

import numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

from aieng import DATA
from aieng.llm import chat

## 1. Index: load documents, embed, store
Corpus: 3,200 Wikipedia passages (`rag-datasets/rag-mini-wikipedia`). Each passage is already a "chunk"; 08.02 covers chunking real documents.

In [ ]:
docs = list(load_dataset("rag-datasets/rag-mini-wikipedia", "text-corpus")["passages"]["passage"])
model = SentenceTransformer("BAAI/bge-small-en-v1.5")
path = DATA / "08-01-index.npy"
if not path.exists():
    np.save(path, model.encode(docs, normalize_embeddings=True, batch_size=64, show_progress_bar=False))
index = np.load(path)
print(index.shape)

## 2. Retrieve
Brute-force cosine search (07.02) is plenty for 3k vectors. bge wants an instruction prefix on queries (07.01).

In [ ]:
def retrieve(question, k=5):
    q = model.encode(["Represent this sentence for searching relevant passages: " + question], normalize_embeddings=True)[0]
    scores = index @ q
    top = np.argsort(-scores)[:k]
    return [(int(i), float(scores[i]), docs[i]) for i in top]


for pid, score, text in retrieve("How old was Lincoln when his mother died?", 3):
    print(f"{score:.3f} [{pid}] {text[:110]}")

## 3. Augment: the prompt
Key choices: number each source so the model can cite it; instruct it to use **only** the sources; give it an explicit way out
("I don't know") — otherwise it fills gaps from memory or invents.

In [ ]:
PROMPT = """Answer the question using only the sources below. Cite the sources you use like [12].
If the sources don't contain the answer, say "I don't know".

{sources}

Question: {question}"""


def build_prompt(question, hits):
    sources = "\n\n".join(f"[{pid}] {text}" for pid, _, text in hits)
    return PROMPT.format(sources=sources, question=question)


print(build_prompt("How old was Lincoln when his mother died?", retrieve("How old was Lincoln when his mother died?", 2))[:700])

## 4. Generate

In [ ]:
def answer(question, k=5):
    hits = retrieve(question, k)
    return chat(build_prompt(question, hits), temperature=0), hits


for q in ["How old was Lincoln when his mother died?", "What is the capital of Uruguay?", "Who won the 2022 FIFA World Cup?"]:
    text, hits = answer(q)
    print(f"Q: {q}\nA: {text}\n   retrieved: {[pid for pid, _, _ in hits]}\n")

The last question isn't in the corpus. A good RAG system says "I don't know" there — even though the model may "know" the answer
from pretraining. Whether you *want* that depends on the product; decide explicitly.

## 5. Evaluate (preview of chapter 10)
Three cheap, automatic checks:
- **Answer accuracy**: does the gold answer string appear in the answer? (crude but useful)
- **Citation validity**: every `[id]` cited was actually retrieved.
- **Abstention**: on questions with no support in the corpus, does it say "I don't know"?

In [ ]:
from aieng import rag  # the same pipeline, packaged (see below)

evalset = rag.eval_set()[:30]
UNANSWERABLE = ["Who won the 2022 FIFA World Cup?", "What is the boiling point of tungsten?", "Who wrote the novel 'One Hundred Years of Solitude'?",
                "What is the population of Tokyo?", "When was the iPhone first released?"]


def cited_ids(text):
    return {int(x) for x in re.findall(r"\[(\d+)\]", text)}


correct = valid = 0
for row in evalset:
    text, hits = answer(row["question"])
    correct += row["answer"].lower().rstrip(".") in text.lower()
    valid += cited_ids(text) <= {pid for pid, _, _ in hits}
abstained = sum("don't know" in answer(q)[0].lower() for q in UNANSWERABLE)
print(f"answer contains gold: {correct}/{len(evalset)} | valid citations: {valid}/{len(evalset)} | abstained: {abstained}/{len(UNANSWERABLE)}")

Some "unanswerable" ones may be partially in the corpus (it's Wikipedia!) — read the outputs before trusting the number.
String-match accuracy undercounts correct paraphrases ("eighteen months" vs "18 months"); chapter 10 adds LLM judges.

## 6. Packaged for later chapters
Chapters 08–11 reuse this exact pipeline from `aieng/rag.py` (plus the mojibake fix from 07, and an `eval_set()` with proxy gold passages):

In [ ]:
print(inspect.getsource(rag.search))
print(inspect.getsource(rag.answer))

## Exercises
1. Index **your own notes or PDFs** (convert to text first; 08.02 does it properly) and ask 10 questions you know the answers to.
2. Vary `k` (1, 3, 5, 10). Plot answer accuracy and prompt tokens. Where's the sweet spot?
3. Make the model quote the exact sentence it relies on before answering (`"Quote: ... Answer: ..."`). Does accuracy or faithfulness change?
4. Swap brute-force retrieval for hybrid + rerank from 07.05 and re-measure.

In [ ]:
# your code here